In [ ]:
# ORIGINAL SCRIPT


"""
Script to estimate the width of rights of way polygons
"""
import numpy as np
import geopandas as gpd
from shapely.geometry import LineString, Point

row = gpd.read_file(r'C:\Users\KyleSteen\Documents\ROW_Width_Analysis\Alaska\Alaska_Exclusion_Analysis_Results_200m.gpkg')
hw = gpd.read_file(r'C:\Users\KyleSteen\Documents\ROW_Width_Analysis\Alaska\NHS_Alaska.gpkg').to_crs(row.crs)


def get_row_width(hw_lines, row, initial_line_length=800, interval=100):
    """
    Generate perpendicular lines along highway centerlines, clip them to ROW polygons,
    save the geometries to GeoPackages, and compute average width.
    """

    # ---------------------------------------------------------
    # Step 1: Explode ROW geometry
    # ---------------------------------------------------------
    row = row.explode(index_parts=True).reset_index(drop=True)
    row["row_id"] = row.index

    # ---------------------------------------------------------
    # Step 2: Reproject highways to ROW CRS
    # ---------------------------------------------------------
    if hw_lines.crs != row.crs:
        hw_lines = hw_lines.to_crs(row.crs)

    # Containers for output
    perpendicular_lines_list = []
    clipped_perpendicular_records = []

    # ---------------------------------------------------------
    # Step 3: Generate perpendicular lines
    # ---------------------------------------------------------
    for _, hw in hw_lines.iterrows():
        line = hw.geometry

        for dist in np.arange(0, line.length, interval):

            # point on the line
            point = line.interpolate(dist)

            # tangent via micro-offsets
            tangent_start = line.interpolate(max(dist - 1e-5, 0))
            tangent_end = line.interpolate(min(dist + 1e-5, line.length))

            dx = tangent_end.x - tangent_start.x
            dy = tangent_end.y - tangent_start.y

            # Perpendicular direction
            perpendicular_dx = -dy
            perpendicular_dy = dx

            norm = np.sqrt(perpendicular_dx**2 + perpendicular_dy**2)
            if norm == 0:
                continue

            perpendicular_dx /= norm
            perpendicular_dy /= norm

            # Build perpendicular segment
            half_length = initial_line_length / 2
            p1 = Point(point.x - perpendicular_dx * half_length, point.y - perpendicular_dy * half_length)
            p2 = Point(point.x + perpendicular_dx * half_length, point.y + perpendicular_dy * half_length)

            perpendicular_line = LineString([p1, p2])

            # Store perpendicular line for output
            perpendicular_lines_list.append(perpendicular_line)

            # ---------------------------------------------------------
            # Step 4: Clip perpendicular line to ROW polygons
            # ---------------------------------------------------------
            for _, poly in row.iterrows():

                clipped = perpendicular_line.intersection(poly.geometry)

                if clipped.is_empty:
                    continue

                # Ensure list of individual line segments
                if clipped.geom_type == "LineString":
                    line_segments = [clipped]
                else:
                    line_segments = [
                        geom for geom in clipped.geoms
                        if geom.geom_type == "LineString"
                    ]

                for segment in line_segments:
                    clipped_perpendicular_records.append({
                        "row_id": poly["row_id"],
                        "length_m": segment.length,
                        "geometry": segment
                    })

    # ---------------------------------------------------------
    # Step 5: Save perpendicular lines to GeoPackage
    # ---------------------------------------------------------
    perpendicular_lines_gdf = gpd.GeoDataFrame(
        geometry=perpendicular_lines_list,
        crs=row.crs
    )
    perpendicular_lines_gdf.to_file("perpendicular_lines.gpkg", driver="GPKG")

    # ---------------------------------------------------------
    # Step 6: Save clipped perpendicular lines
    # ---------------------------------------------------------
    if len(clipped_perpendicular_records) == 0:
        row["approx_length_meters"] = 0
        return row

    clipped_perpendicular_gdf = gpd.GeoDataFrame(
        clipped_perpendicular_records,
        geometry="geometry",
        crs=row.crs
    )
    clipped_perpendicular_gdf.to_file("clipped_perpendicular_lines.gpkg", driver="GPKG")

    # ---------------------------------------------------------
    # Step 7: Compute average clipped perpendicular length per ROW polygon
    # ---------------------------------------------------------
    grouped = clipped_perpendicular_gdf.groupby("row_id")["length_m"].mean()
    row = row.copy()
    row["approx_length_meters"] = row["row_id"].map(grouped).fillna(0)

    row.to_file("row_with_widths.gpkg", driver="GPKG")

    return row

get_row_width(hw_lines=hw, row=row, initial_line_length=1200, interval=100)


In [3]:
# Alaska Final Script Modified - No Explode (ROW_ID already unique)
# Spatial Index & Writes to CSV

"""
Script to estimate the width of rights of way polygons.
Optimized for large areas using spatial indexing and logging.
Outputs ROW widths to CSV, but keeps perpendicular lines as GPKG.
Assumes ROW_ID is already unique (Multipart to Singlepart already run).
"""

import numpy as np
import geopandas as gpd
from shapely.geometry import LineString, Point
import logging
from tqdm import tqdm

# -----------------------------
# Setup logging
# -----------------------------
logging.basicConfig(
    filename="row_width_log.txt",
    filemode="w",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
console.setFormatter(formatter)
logging.getLogger("").addHandler(console)

# -----------------------------
# Load data
# -----------------------------
row = gpd.read_file(
    r"C:\Users\KyleSteen.AzureAD\Documents\ROW_Width_Analysis\Alaska\Alaska_3_10.gpkg"
)

hw = gpd.read_file(
    r"C:\Users\KyleSteen.AzureAD\Documents\ROW_Width_Analysis\Alaska\NHS_Alaska.gpkg"
).to_crs(row.crs)

# -----------------------------
# Function
# -----------------------------
def get_row_width(hw_lines, row, initial_line_length=1000, interval=10, output_csv="row_with_widths.csv"):
    """
    Generate perpendicular lines along highway centerlines, clip them to ROW polygons,
    save the geometries to GeoPackages, and compute average width.
    ROW widths are written to a CSV.
    """

    # Ensure CRS match
    if hw_lines.crs != row.crs:
        hw_lines = hw_lines.to_crs(row.crs)

    # Create spatial index
    logging.info("Creating spatial index for ROW polygons...")
    sindex = row.sindex

    perpendicular_lines_list = []
    clipped_perpendicular_records = []

    logging.info(f"Processing {len(hw_lines)} highway lines...")

    for hw_idx, hw_feature in hw_lines.iterrows():
        line = hw_feature.geometry
        logging.info(f"Processing highway {hw_idx+1}/{len(hw_lines)} (length {line.length:.0f} m)")

        for dist in tqdm(np.arange(0, line.length, interval), desc=f"HW {hw_idx+1}", leave=False):

            point = line.interpolate(dist)

            tangent_start = line.interpolate(max(dist - 1e-5, 0))
            tangent_end = line.interpolate(min(dist + 1e-5, line.length))

            dx = tangent_end.x - tangent_start.x
            dy = tangent_end.y - tangent_start.y

            perp_dx = -dy
            perp_dy = dx

            norm = np.sqrt(perp_dx**2 + perp_dy**2)
            if norm == 0:
                continue

            perp_dx /= norm
            perp_dy /= norm

            half_length = initial_line_length / 2
            p1 = Point(point.x - perp_dx * half_length, point.y - perp_dy * half_length)
            p2 = Point(point.x + perp_dx * half_length, point.y + perp_dy * half_length)

            perpendicular_line = LineString([p1, p2])
            perpendicular_lines_list.append(perpendicular_line)

            # Spatial index filter
            possible_matches_index = list(sindex.intersection(perpendicular_line.bounds))
            if not possible_matches_index:
                continue

            possible_matches = row.iloc[possible_matches_index]

            for _, poly in possible_matches.iterrows():
                clipped = perpendicular_line.intersection(poly.geometry)
                if clipped.is_empty:
                    continue

                if clipped.geom_type == "LineString":
                    segments = [clipped]
                else:
                    segments = [
                        geom for geom in clipped.geoms
                        if geom.geom_type == "LineString"
                    ]

                for segment in segments:
                    clipped_perpendicular_records.append({
                        "ROW_ID": poly["ROW_ID"],
                        "length_m": segment.length,
                        "geometry": segment
                    })

    # -----------------------------
    # Save perpendicular lines
    # -----------------------------
    logging.info("Saving all perpendicular lines to GPKG...")
    gpd.GeoDataFrame(
        geometry=perpendicular_lines_list,
        crs=row.crs
    ).to_file("perpendicular_lines.gpkg", driver="GPKG")

    # -----------------------------
    # Process clipped segments
    # -----------------------------
    if len(clipped_perpendicular_records) == 0:
        logging.warning("No clipped perpendicular lines found.")
        row["Approximate_Width_Meters"] = 0
    else:
        logging.info("Saving clipped perpendicular lines to GPKG...")
        clipped_gdf = gpd.GeoDataFrame(
            clipped_perpendicular_records,
            geometry="geometry",
            crs=row.crs
        )

        clipped_gdf.to_file("clipped_perpendicular_lines.gpkg", driver="GPKG")

        logging.info("Calculating average ROW widths...")
        grouped = clipped_gdf.groupby("ROW_ID")["length_m"].mean()

        row["Approximate_Width_Meters"] = (
            row["ROW_ID"].map(grouped).fillna(0)
        )

    # -----------------------------
    # Export ONLY requested columns
    # -----------------------------
    logging.info(f"Writing ROW widths to CSV: {output_csv}")

    export_df = row[[
        "ROW_ID",
        "Square_Met",
        "Approximate_Width_Meters"
    ]]

    export_df.to_csv(output_csv, index=False)

    logging.info("Process completed successfully!")
    return row


# -----------------------------
# Run the function
# -----------------------------
get_row_width(
    hw_lines=hw,
    row=row,
    initial_line_length=1000,
    interval=10
)

2026-03-10 12:04:59,932 - INFO - Creating spatial index for ROW polygons...
2026-03-10 12:04:59,935 - INFO - Processing 1 highway lines...
2026-03-10 12:04:59,937 - INFO - Processing highway 1/1 (length 3588517 m)
2026-03-10 13:12:43,151 - INFO - Saving all perpendicular lines to GPKG...                                             
2026-03-10 13:12:44,648 - INFO - Created 358,852 records
2026-03-10 13:12:45,234 - INFO - Saving clipped perpendicular lines to GPKG...
2026-03-10 13:12:45,671 - INFO - Created 77,283 records
2026-03-10 13:12:45,838 - INFO - Calculating average ROW widths...
2026-03-10 13:12:45,848 - INFO - Writing ROW widths to CSV: row_with_widths.csv
2026-03-10 13:12:45,855 - INFO - Process completed successfully!


,State_Name,Square_Met,Shape_Leng,ROW_ID,geometry,Approximate_Width_Meters
0,Alaska,3968.124350,382.337729,1.0,"MULTIPOLYGON (((1009505.884 1184950.813, 10095...",22.144332
1,Alaska,6612.099889,902.150732,2.0,"MULTIPOLYGON (((238479.283 1274822.957, 238475...",15.401476
2,Alaska,9734.752760,1282.895522,3.0,"MULTIPOLYGON (((205601.989 1334923.369, 205601...",15.440505
3,Alaska,21647.719351,1126.098320,4.0,"MULTIPOLYGON (((205451.898 1334856.087, 205451...",40.862152
4,Alaska,14787.840330,834.112877,5.0,"MULTIPOLYGON (((205693.623 1335170.735, 205693...",38.381547
...,...,...,...,...,...,...
2698,Alaska,1114.809767,157.933017,2699.0,"MULTIPOLYGON (((252243.626 1287788.347, 252243...",16.647531
2699,Alaska,5846.793264,615.130566,2700.0,"MULTIPOLYGON (((250502.418 1296284.998, 250502...",34.888304
2700,Alaska,8098.364367,523.058530,2701.0,"MULTIPOLYGON (((250676.641 1295855.316, 250677...",34.104760
2701,Alaska,1261.560102,182.958682,2702.0,"MULTIPOLYGON (((250488.952 1296044.265, 250489...",16.321583
